# MLB batted balls — EDA + home-run classifier

Reproduces the exit-velocity / launch-angle geometry from the reference blog
([Tyler James Burch hit classifier](https://tylerjamesburch.com/blog/baseball/hit-classifier-3)),
then trains an XGBoost model to predict **home runs** — the rare, desirable
minority class — on **all** training data (no majority-class downselection).
See `docs/plan.md` 3.2.

The plotting helpers live in `negative_ml.viz.baseball`, so this notebook and
`scripts/baseball_eda_model.py` draw exactly the same figures.

In [ ]:
import matplotlib.pyplot as plt

from negative_ml import eval as ev
from negative_ml.domains import baseball
from negative_ml.models import make_model
from negative_ml.splitting import random_split
from negative_ml.viz import baseball as viz

# First call pulls the 2024 Statcast season via pybaseball (cached under data/raw).
ds = baseball.load()
n_pos = int(ds.y.sum())
print(f"{ds.n_samples:,} batted balls; {n_pos:,} home runs ({ds.y.mean():.2%} minority)")
ds.meta["outcome"].value_counts()

## 1. Class balance

Most batted balls are outs; home runs are a small minority — the imbalance this project studies.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
viz.plot_class_balance(ds.meta, ax)
plt.show()

## 2. Batted-ball geometry (reproduce the blog)

In the exit-velocity / launch-angle plane the outcome classes separate cleanly:
home runs occupy a tight high-exit-velocity, mid-launch-angle cluster (~95+ mph,
~12-50 deg). The adjusted spray angle is pull-positive and shows the expected
pull skew.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
viz.plot_exit_velo_launch_angle(ds, ax)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
viz.plot_spray_distribution(ds, axes[0])
viz.plot_hr_density(ds, axes[1])
plt.tight_layout()
plt.show()

## 3. Train an XGBoost home-run classifier (all data)

Stratified random split, then fit on the **entire** training partition — no
selection strategies yet. This confirms the baseball data and model pipeline work
end to end before any downselection experiments.

In [ ]:
train_idx, test_idx = random_split(ds, test_size=0.2)
model = make_model("classification")
model.fit(ds.X[train_idx], ds.y[train_idx])

y_test = ds.y[test_idx]
y_score = model.predict_proba(ds.X[test_idx])[:, 1]
ev.compute_metrics(y_test, y_score)

## 4. Model performance

Home runs are nearly linearly separable in this feature space, so global
discrimination is very strong (AUROC ~0.99). The predicted-probability heatmap
lights up the same region as the home-run cluster above.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
viz.plot_roc(y_test, y_score, axes[0])
viz.plot_pr(y_test, y_score, axes[1])
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
viz.plot_confusion(y_test, y_score, axes[0])
viz.plot_precision_at_k(y_test, y_score, axes[1])
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
viz.plot_feature_importance(model, ds.feature_names, axes[0])
viz.plot_probability_heatmap(model, ds, axes[1])
plt.tight_layout()
plt.show()